# MuRIL fine-tuning (Colab) - English + Hinglish hate speech

**Status: not run yet. No MuRIL results exist until you run this notebook on a GPU and the verification cell passes.**

### Before you start (on your Mac)
```
cd ~/arya/Programming/NLP
zip -r project_src.zip src train_muril.py -x "*/__pycache__/*" "*.pyc"
```
Then in Colab: **Runtime > Change runtime type > T4 GPU**, and run the cells below **in order**.

### What to expect
- Training data: English = Davidson tweets (downloaded inside Colab), Hinglish = the 15-sentence **synthetic** seed (9 training rows).
- The script prints a loud warning and records it in `label_config.json`. **Do not treat this model as a Hinglish hate-speech detector.** English metrics are the meaningful ones; Hinglish test n=3 is anecdotal.
- Nothing is installed on your Mac by this notebook. Dependencies install inside Colab only.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then rerun'
print('GPU:', torch.cuda.get_device_name(0))
# torch is preinstalled on Colab; only add what is missing. (Runs inside Colab, not on your Mac.)
!pip install -q 'transformers>=4.41' accelerate emoji scikit-learn pandas
import transformers; print('transformers', transformers.__version__)

In [ ]:
from google.colab import files
up = files.upload()   # choose project_src.zip
assert 'project_src.zip' in up, 'upload project_src.zip'
!rm -rf project && unzip -q -o project_src.zip -d project
%cd project
!ls -la . src

In [ ]:
# Pre-flight: imports, label mapping, preprocessing hash, leakage-free splits (no training yet)
import sys, hashlib; sys.path.insert(0, 'src')
from preprocess import ID_TO_LABEL, LABEL_TO_ID, clean_text
assert ID_TO_LABEL == {0: 'HATE', 1: 'OFFENSIVE', 2: 'NORMAL'}, ID_TO_LABEL
print('preprocess sha256:', hashlib.sha256(open('src/preprocess.py','rb').read()).hexdigest())
assert 'https' not in clean_text('see https://x.com/a now'), 'old preprocessing (URL bug) uploaded - re-zip from the Mac'
from data import build_splits
stats = {}
tr, va, te = build_splits(stats)
print(stats)
for n, d in (('train', tr), ('val', va), ('test', te)):
    print(n, len(d), d.groupby(['language', 'label']).size().to_dict())
assert not (set(tr.clean_text) & set(va.clean_text)) and not (set(tr.clean_text) & set(te.clean_text)) and not (set(va.clean_text) & set(te.clean_text))
print('OK: no overlap between splits on normalised text')

In [ ]:
# Optional: real Hinglish data you are licensed to use (e.g. HASOC CSVs with task_2 labels HATE/OFFN/PRFN/NONE).
# Leave HASOC = '' to train on the synthetic seed only (see warning above).
# up2 = files.upload(); HASOC = ' '.join(up2.keys())
HASOC = ''

In [ ]:
# Train. Selection uses validation macro-F1 only; test is evaluated once at the end.
!python train_muril.py --out models/muril_final_model --epochs 4 --hasoc $HASOC

In [ ]:
# VERIFY the artifacts exactly as the Streamlit app will (validation + real GPU/CPU inference)
import json, numpy as np
import muril_model
ok, msg, cfg = muril_model.check_artifacts('models/muril_final_model')
print('artifact check:', ok, msg); assert ok, msg
tok, model, torch = muril_model.load('models/muril_final_model')
fn = muril_model.make_proba_fn(tok, model, torch, cfg['max_len'])
samples = ['Have a lovely day my friend', 'you are all idiots and I hate you', 'kal match bahut mast tha bhai', 'tu pagal hai bsdk chup reh']
P = fn([clean_text(t) for t in samples])
assert P.shape == (len(samples), 3) and np.allclose(P.sum(1), 1, atol=1e-4)
for t, p in zip(samples, P):
    print(f'{ID_TO_LABEL[int(p.argmax())]:9s} {np.round(p, 3).tolist()}  <- {t}')
res = json.load(open('models/muril_final_model/eval_results.json'))
print('\nvalidation (best checkpoint):', res.get('validation_best_checkpoint'))
for k in ('combined', 'english', 'hinglish'):
    if k in res: print(f"{k:9s} n={res[k]['n']:4d} macro_f1={res[k]['macro_f1']:.4f} reliable={res[k]['reliable']}")
print('\nlimitations:', cfg.get('limitations'))

In [ ]:
# Package ONLY final artifacts (no checkpoints), list the zip, download.
!rm -rf models/muril_final_model/checkpoints
!cd models && rm -f ../muril_final_model.zip && zip -r ../muril_final_model.zip muril_final_model
!unzip -l muril_final_model.zip
files.download('muril_final_model.zip')

### After download (on your Mac)
Tell Claude: *"muril_final_model.zip is in Downloads."* It will list the ZIP, extract into `NLP/models/muril_final_model/`, ask before installing `requirements-muril.txt` into `.venv312`, run the tests, run a real prediction and restart Streamlit.
Manual equivalent: `unzip ~/Downloads/muril_final_model.zip -d ~/arya/Programming/NLP/models/`